In [2]:
import pandas as pd

file_path = '/home/mohnish/amazon_sentiment_analysis/data/Cleaned_Amazon_Reviews_Sample.csv'
df = pd.read_csv(file_path)
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (91939, 9)


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=2500)

X = tfidf.fit_transform(df['Final_Clean_Text'])
y = df['Sentiment']

print(f"Feature Matrix Shape: {X.shape}") 

Feature Matrix Shape: (91939, 2500)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 73551
Test set size: 18388


In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix, average_precision_score

balanced_model = LogisticRegression(class_weight='balanced', max_iter=20000, random_state=42)
balanced_model.fit(X_train, y_train)

y_scores = balanced_model.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_scores)

f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
best_threshold = thresholds[np.argmax(f1_scores)]

y_pred_final = (y_scores >= best_threshold).astype(int)

print(f"Optimal Threshold (F1-Maximized): {best_threshold:.4f}")
print("\nFinal Classification Report (Balanced Loss + Tuned Threshold):")
print(classification_report(y_test, y_pred_final))
print(f"AUPRC (Area Under Precision-Recall Curve): {average_precision_score(y_test, y_scores):.4f}")

Optimal Threshold (F1-Maximized): 0.1989

Final Classification Report (Balanced Loss + Tuned Threshold):
              precision    recall  f1-score   support

           0       0.80      0.61      0.69      2977
           1       0.93      0.97      0.95     15411

    accuracy                           0.91     18388
   macro avg       0.86      0.79      0.82     18388
weighted avg       0.91      0.91      0.91     18388

AUPRC (Area Under Precision-Recall Curve): 0.9870


In [6]:
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, classification_report, average_precision_score
import numpy as np

pos_count = (y_train == 1).sum()
neg_count = (y_train == 0).sum()
ratio = neg_count / pos_count

xgb_balanced = XGBClassifier(
    scale_pos_weight=ratio, 
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
)


xgb_balanced.fit(X_train, y_train)


xgb_scores = xgb_balanced.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, xgb_scores)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
best_threshold_xgb = thresholds[np.argmax(f1_scores)]

y_pred_xgb = (xgb_scores >= best_threshold_xgb).astype(int)

print(f"XGBoost Optimal Threshold: {best_threshold_xgb:.4f}")
print("\nFinal Classification Report (XGBoost Balanced + Tuned):")
print(classification_report(y_test, y_pred_xgb))
print(f"AUPRC: {average_precision_score(y_test, xgb_scores):.4f}")

XGBoost Optimal Threshold: 0.2735

Final Classification Report (XGBoost Balanced + Tuned):
              precision    recall  f1-score   support

           0       0.79      0.45      0.57      2977
           1       0.90      0.98      0.94     15411

    accuracy                           0.89     18388
   macro avg       0.85      0.71      0.76     18388
weighted avg       0.88      0.89      0.88     18388

AUPRC: 0.9777


In [7]:
import pandas as pd
import matplotlib.pyplot as plt

feature_names = tfidf.get_feature_names_out()

coefficients = balanced_model.coef_[0]

word_importance = pd.DataFrame({
    'Word': feature_names,
    'Weight': coefficients
})

top_negative = word_importance.sort_values(by='Weight').head(20)
top_positive = word_importance.sort_values(by='Weight', ascending=False).head(20)

print("--- TOP 20 NEGATIVE INDICATORS ---")
print(top_negative[['Word', 'Weight']])

print("\n--- TOP 20 POSITIVE INDICATORS ---")
print(top_positive[['Word', 'Weight']])

--- TOP 20 NEGATIVE INDICATORS ---
                Word    Weight
625     disappointed -8.923508
626    disappointing -7.105608
2345   unfortunately -6.613558
1069        horrible -6.554871
1830          return -6.474372
2235        terrible -6.234160
131            awful -6.227998
206            bland -6.093979
627   disappointment -5.837622
136              bad -5.318275
1066             hop -5.164355
298           cancel -4.834101
2420            weak -4.708538
2491            yuck -4.658140
2413           waste -4.486901
2215       tasteless -4.475561
1951           shame -4.472789
2041           sorry -4.452179
2089           stale -4.401252
1403           money -4.365482

--- TOP 20 POSITIVE INDICATORS ---
           Word     Weight
962       great  10.874958
569   delicious   9.470241
1580    perfect   8.784571
1295       love   8.092234
938        good   7.829917
2460  wonderful   7.485297
741   excellent   7.397321
130     awesome   6.851068
1048     highly   6.626843
69      

In [8]:

y_probs = balanced_model.predict_proba(X_test)[:, 1]
y_pred = (y_probs >= 0.2276).astype(int)

results_df = pd.DataFrame({
    'Text': df.loc[y_test.index, 'Text'], 
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Confidence': y_probs
})

errors = results_df[(results_df['Actual'] == 0) & (results_df['Predicted'] == 1)]

print(f"Total Missed Negative Reviews: {len(errors)}")
display(errors.head(5))

Total Missed Negative Reviews: 1058


,Text,Actual,Predicted,Confidence
64918,"Although I love the Nana's brownie mint, and p...",0,1,0.411324
78624,about 1 in 10 pieces actually taste like teriy...,0,1,0.824771
8665,They kept raising the price. The product was g...,0,1,0.870835
71299,The Switch products want you to believe that t...,0,1,0.902933
65628,"Attractive bottle, smells nice, thick enough (...",0,1,0.595026
